In [6]:


import os
import torch
import numpy as np
from PIL import Image
from transformers import SamProcessor, SamModel
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_PATH = "/content/drive/MyDrive/Segmentation Dataset/sam_checkpoints/sam_base_encoder_epoch_100.pth"
IMAGE_DIR = "/content/drive/MyDrive/Segmentation Dataset/images"
OUTPUT_DIR = "/content/drive/MyDrive/Segmentation Dataset/predicted_masks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading model")
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
model = SamModel.from_pretrained("facebook/sam-vit-base")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict, strict=False)

model.to(DEVICE)
model.eval()
print("Model loaded successfully!")

def get_bounding_box_from_mask(mask_np):
    ys, xs = np.where(mask_np > 0)
    if ys.size == 0:
        return [0, 0, mask_np.shape[1], mask_np.shape[0]]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    return [int(x_min), int(y_min), int(x_max), int(y_max)]

def segment_images(image_paths):
    results = []

    for path in image_paths:
        image_name = os.path.basename(path)
        print(f"\nSegmenting {image_name} ...")

        image = Image.open(path).convert("RGB")

        bbox = [0, 0, image.size[0], image.size[1]]
        inputs = processor(image, input_boxes=[[bbox]], return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs, multimask_output=False)
            mask_pred = torch.sigmoid(outputs.pred_masks[0, 0]).cpu().numpy()

        mask_bin = (mask_pred > 0.5).astype(np.uint8) * 255

        mask_image = Image.fromarray(mask_bin.squeeze().astype(np.uint8), mode="L")

        save_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(image_name)[0]}_mask.png")
        mask_image.save(save_path)
        results.append(save_path)

        print(f"Saved mask to {save_path}")

    return results

if __name__ == "__main__":
    all_images = [
        os.path.join(IMAGE_DIR, f)
        for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]

    if not all_images:
        raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

    print(f"Found {len(all_images)} images for segmentation.")
    segmented = segment_images(all_images)
    print(f"\n✅ Segmentation complete for {len(segmented)} images.")
    print("Output directory:", OUTPUT_DIR)


Mounted at /content/drive
Loading model


KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
path = "/content/drive/MyDrive/Segmentation Dataset/sam_checkpoints/sam_base_encoder_epoch_100.pth"

print("Exists:", os.path.exists(path))
!ls -lh "/content/drive/MyDrive/Segmentation Dataset/sam_checkpoints/"


Mounted at /content/drive
Exists: True
total 3.9G
-rw------- 1 root root 358M Nov 10 20:23 sam_base_encoder_epoch_100.pth
-rw------- 1 root root 358M Nov 10 15:03 sam_base_encoder_epoch_10.pth
-rw------- 1 root root 358M Nov 10 15:39 sam_base_encoder_epoch_20.pth
-rw------- 1 root root 358M Nov 10 16:15 sam_base_encoder_epoch_30.pth
-rw------- 1 root root 358M Nov 10 16:51 sam_base_encoder_epoch_40.pth
-rw------- 1 root root 358M Nov 10 17:26 sam_base_encoder_epoch_50.pth
-rw------- 1 root root 358M Nov 10 18:02 sam_base_encoder_epoch_60.pth
-rw------- 1 root root 358M Nov 10 18:37 sam_base_encoder_epoch_70.pth
-rw------- 1 root root 358M Nov 10 19:13 sam_base_encoder_epoch_80.pth
-rw------- 1 root root 358M Nov 10 19:48 sam_base_encoder_epoch_90.pth
-rw------- 1 root root 358M Nov 10 20:23 sam_base_encoder_final.pth
-rw------- 1 root root  34K Sep 10  2024 segmentation_model.tflite
